In [27]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load your CSV
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data_with_better_index 2.csv")  # update if your file path changes

# Rename for consistency
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "pct_highschool_or_more_(2023)": "education_2022",
    "Pop 2023": "pop_2023",
    "Established firms 2022": "firms_2022"
})

# Convert relevant columns to numeric
cols_to_numeric = [
    'church_persistence_index',
    'income_1989_real_2023',
    'pct_hs_1990',
    'pop_2023',
    'firms_2022',
    'education_2022'
]
df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

# Drop missing values
df = df.dropna(subset=cols_to_numeric)

# Log-transform population
df['log_pop_2023'] = np.log(df['pop_2023'])

# Standardize controls
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run the regression
edu_model = smf.ols(
    formula='education_2022 ~ church_persistence_index + income_1989_scaled + log_pop_2023 + C(State)',
    data=df
).fit(cov_type='HC3')

# Output summary
print(edu_model.summary())

                            OLS Regression Results                            
Dep. Variable:         education_2022   R-squared:                       0.465
Model:                            OLS   Adj. R-squared:                  0.456
Method:                 Least Squares   F-statistic:                     63.71
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        12:44:10   Log-Likelihood:                -8380.2
No. Observations:                3004   AIC:                         1.686e+04
Df Residuals:                    2953   BIC:                         1.717e+04
Df Model:                          50                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               